# Exploration actuarielle du portefeuille espagnol

Ce notebook documente une exploration locale : structure des contrats, qualité des données, cohérence des mesures et indicateurs de sinistralité. Les colonnes et hypothèses utilisées sont propres à cette base ; le notebook ne constitue pas le moteur généraliste de l’application V2.

Les calculs sont effectués en Python, sans appel au modèle IA. Les données restent locales et les sorties ne sont pas incluses dans la version destinée au partage.


In [ ]:
import pandas as pd
from pathlib import Path

# Chemin du fichier
fichier = Path("../data/portefeuille.csv")

# Importation
df = pd.read_csv(fichier, sep=None, engine="python")

In [ ]:
print("Nombre de lignes :", df.shape[0])
print("Nombre de colonnes :", df.shape[1])
display(df.head())

In [ ]:
# Liste des variables et de leurs types
variables = pd.DataFrame({
    "Variable": df.columns,
    "Type": df.dtypes.astype(str).values,
    "Valeurs manquantes (%)": (
        df.isna().mean().values * 100
    ).round(2),
    "Valeurs distinctes": df.nunique().values
})

display(variables)

In [ ]:
# 1. Nombre total d'observations
print("Nombre total de lignes :", len(df))

# 2. Répartition par année
print("\nRépartition par année :")
display(
    df["year"].value_counts()
    .sort_index()
    .rename_axis("Année")
    .reset_index(name="Nombre de lignes")
)

# 3. Nombre d'assurés distincts par année
print("\nNombre d'assurés distincts par année :")
display(
    df.groupby("year")["insured_id"].nunique()
)

# 4. Nombre d'années observées par assuré
print("\nNombre d'années par assuré :")
display(
    df.groupby("insured_id")["year"]
    .nunique()
    .value_counts()
    .sort_index()
)

# 5. Doublons sur le couple assuré-année
print(
    "\nDoublons assuré-année :",
    df.duplicated(subset=["insured_id", "year"]).sum()
)

In [ ]:
# Statistiques descriptives des expositions
display(
    df[["total_exposure", "liability_exposure"]]
    .describe()
)

# Nombre de valeurs nulles ou négatives
for col in ["total_exposure", "liability_exposure"]:
    print(f"\n{col}")
    print("Expositions nulles :", (df[col] == 0).sum())
    print("Expositions négatives :", (df[col] < 0).sum())
    print("Expositions > 1 :", (df[col] > 1).sum())

# Comparaison des deux variables
print(
    "\nExpositions différentes :",
    (df["total_exposure"] != df["liability_exposure"]).sum()
)

# Exposition totale par année
display(
    df.groupby("year")[
        ["total_exposure", "liability_exposure"]
    ].sum()
)

In [ ]:
# Statistiques descriptives des sinistres et des coûts
display(
    df[["total_claims", "total_incurred"]].describe()
)

# Distribution du nombre de sinistres
print("Distribution du nombre de sinistres :")
display(
    df["total_claims"]
    .value_counts()
    .sort_index()
)

# Contrôles de cohérence
print("Nombre de sinistres négatifs :",
      (df["total_claims"] < 0).sum())

print("Coûts négatifs :",
      (df["total_incurred"] < 0).sum())

print("Sinistres positifs avec coût nul :",
      ((df["total_claims"] > 0) &
       (df["total_incurred"] == 0)).sum())

print("Aucun sinistre avec coût non nul :",
      ((df["total_claims"] == 0) &
       (df["total_incurred"] != 0)).sum())

print("Sinistres sur exposition nulle :",
      ((df["total_exposure"] == 0) &
       (df["total_claims"] > 0)).sum())

In [ ]:
# Synthèse par année
bilan = df.groupby("year").agg(
    observations=("insured_id", "size"),
    exposition=("total_exposure", "sum"),
    nb_sinistres=("total_claims", "sum"),
    charge_sinistres=("total_incurred", "sum"),
    primes=("total_premium", "sum")
)

# Indicateurs actuariels
bilan["frequence"] = (
    bilan["nb_sinistres"] / bilan["exposition"]
)

bilan["cout_moyen"] = (
    bilan["charge_sinistres"] / bilan["nb_sinistres"]
)

bilan["prime_pure"] = (
    bilan["charge_sinistres"] / bilan["exposition"]
)

bilan["prime_moyenne"] = (
    bilan["primes"] / bilan["exposition"]
)

bilan["ratio_SP"] = (
    bilan["charge_sinistres"] / bilan["primes"]
)

# Ajout du total portefeuille
total = bilan[
    ["observations", "exposition", "nb_sinistres",
     "charge_sinistres", "primes"]
].sum()

total.name = "TOTAL"

bilan = pd.concat([bilan, total.to_frame().T])

# Recalcul des ratios sur les totaux
for idx in ["TOTAL"]:
    bilan.loc[idx, "frequence"] = (
        bilan.loc[idx, "nb_sinistres"] /
        bilan.loc[idx, "exposition"]
    )
    bilan.loc[idx, "cout_moyen"] = (
        bilan.loc[idx, "charge_sinistres"] /
        bilan.loc[idx, "nb_sinistres"]
    )
    bilan.loc[idx, "prime_pure"] = (
        bilan.loc[idx, "charge_sinistres"] /
        bilan.loc[idx, "exposition"]
    )
    bilan.loc[idx, "prime_moyenne"] = (
        bilan.loc[idx, "primes"] /
        bilan.loc[idx, "exposition"]
    )
    bilan.loc[idx, "ratio_SP"] = (
        bilan.loc[idx, "charge_sinistres"] /
        bilan.loc[idx, "primes"]
    )

display(bilan.round(3))

In [ ]:
# Colonnes détaillées de sinistres
claims_cols = [
    "liability_claims",
    "liability_property_claims",
    "liability_injury_claims",
    "property_claims",
    "theft_claims",
    "fire_claims",
    "glass_claims",
    "legal_protection_claims",
    "occupants_claims"
]

# Colonnes détaillées de charges
incurred_cols = [
    "liability_incurred",
    "liability_property_incurred",
    "liability_injury_incurred",
    "property_incurred",
    "theft_incurred",
    "fire_incurred",
    "glass_incurred",
    "legal_protection_incurred",
    "occupants_incurred"
]

# Vérification des totaux
print("Sinistres :")
print("Total déclaré :", df["total_claims"].sum())
print("Somme des détails :", df[claims_cols].sum().sum())

print("\nCharges :")
print("Total déclaré :", df["total_incurred"].sum())
print("Somme des détails :", df[incurred_cols].sum().sum())

# Comparaison au niveau de chaque ligne
print("\nLignes avec écart sur le nombre de sinistres :")
print(
    (df["total_claims"] != df[claims_cols].sum(axis=1)).sum()
)

print("\nLignes avec écart sur les charges (> 0,01) :")
print(
    (
        (df["total_incurred"] -
         df[incurred_cols].sum(axis=1)).abs() > 0.01
    ).sum()
)

In [ ]:
# Variables de sinistres sans les sous-catégories RC
claims_main = [
    "liability_claims",
    "property_claims",
    "theft_claims",
    "fire_claims",
    "glass_claims",
    "legal_protection_claims",
    "occupants_claims"
]

# Variables de charges sans les sous-catégories RC
incurred_main = [
    "liability_incurred",
    "property_incurred",
    "theft_incurred",
    "fire_incurred",
    "glass_incurred",
    "legal_protection_incurred",
    "occupants_incurred"
]

# Comparaison des nombres de sinistres
ecart_claims = (
    df["total_claims"] -
    df[claims_main].sum(axis=1)
)

# Comparaison des charges
ecart_incurred = (
    df["total_incurred"] -
    df[incurred_main].sum(axis=1)
)

print("=== NOMBRE DE SINISTRES ===")
print("Écart global :", ecart_claims.sum())
print("Lignes avec écart :", (ecart_claims != 0).sum())

print("\n=== CHARGES DE SINISTRES ===")
print("Écart global :", ecart_incurred.sum())
print(
    "Lignes avec écart > 0,01 :",
    (ecart_incurred.abs() > 0.01).sum()
)

# Vérification des sous-catégories RC
print("\n=== DÉCOMPOSITION RC ===")

print(
    "Lignes où RC matériel + corporel != RC total :",
    (
        df["liability_property_claims"] +
        df["liability_injury_claims"] !=
        df["liability_claims"]
    ).sum()
)

print(
    "Lignes où charges RC matériel + corporel != RC total :",
    (
        (
            df["liability_property_incurred"] +
            df["liability_injury_incurred"] -
            df["liability_incurred"]
        ).abs() > 0.01
    ).sum()
)

In [ ]:
garanties = {
    "Responsabilité civile": ("liability_claims", "liability_incurred"),
    "Dommages": ("property_claims", "property_incurred"),
    "Vol": ("theft_claims", "theft_incurred"),
    "Incendie": ("fire_claims", "fire_incurred"),
    "Bris de glace": ("glass_claims", "glass_incurred"),
    "Protection juridique": (
        "legal_protection_claims",
        "legal_protection_incurred"
    ),
    "Occupants": ("occupants_claims", "occupants_incurred")
}

resultats = []

for garantie, (col_nb, col_cout) in garanties.items():
    nb = df[col_nb].sum()
    cout = df[col_cout].sum()

    resultats.append({
        "Garantie": garantie,
        "Nombre de sinistres": nb,
        "Charge totale": cout,
        "Coût moyen": cout / nb if nb > 0 else 0,
        "Part des sinistres (%)": nb / df["total_claims"].sum() * 100,
        "Part des charges (%)": cout / df["total_incurred"].sum() * 100
    })

bilan_garanties = pd.DataFrame(resultats)

display(
    bilan_garanties
    .sort_values("Charge totale", ascending=False)
    .round(2)
)

In [ ]:
premium_cols = [
    "liability_premium",
    "property_damage_premium",
    "theft_premium",
    "fire_premium",
    "glass_premium",
    "legal_protection_premium",
    "occupants_premium"
]

# Somme des primes détaillées
somme_primes = df[premium_cols].sum(axis=1)

# Écart avec la prime totale
ecart = df["total_premium"] - somme_primes

print("Prime totale déclarée :", df["total_premium"].sum())
print("Somme des primes détaillées :", somme_primes.sum())

print("\nÉcart global :", ecart.sum())

print(
    "Nombre de lignes avec écart > 0,01 :",
    (ecart.abs() > 0.01).sum()
)

print("\nPrimes négatives :")
display((df[premium_cols + ["total_premium"]] < 0).sum())

In [ ]:
# Correspondance entre garanties, primes et charges
garanties_sp = {
    "RC": ("liability_premium", "liability_incurred"),
    "Dommages": ("property_damage_premium", "property_incurred"),
    "Vol": ("theft_premium", "theft_incurred"),
    "Incendie": ("fire_premium", "fire_incurred"),
    "Bris de glace": ("glass_premium", "glass_incurred"),
    "Protection juridique": (
        "legal_protection_premium",
        "legal_protection_incurred"
    ),
    "Occupants": ("occupants_premium", "occupants_incurred")
}

resultats = []

for annee, groupe in df.groupby("year"):
    primes_totales = groupe["total_premium"].sum()

    for garantie, (col_prime, col_charge) in garanties_sp.items():
        primes = groupe[col_prime].sum()
        charges = groupe[col_charge].sum()

        resultats.append({
            "Année": annee,
            "Garantie": garantie,
            "Primes": primes,
            "Charges": charges,
            "S/P (%)": charges / primes * 100 if primes > 0 else np.nan,
            "Contribution S/P (points)": charges / primes_totales * 100
        })

sp_garanties = pd.DataFrame(resultats)

# Tableau des S/P par garantie et par année
tableau_sp = sp_garanties.pivot(
    index="Garantie",
    columns="Année",
    values="S/P (%)"
)

display(tableau_sp.round(2))

# Décomposition de l'évolution du S/P global
contributions = sp_garanties.pivot(
    index="Garantie",
    columns="Année",
    values="Contribution S/P (points)"
)

contributions["Variation 2022-2024 (points)"] = (
    contributions[2024] - contributions[2022]
)

display(
    contributions.sort_values(
        "Variation 2022-2024 (points)",
        ascending=False
    ).round(2)
)

print(
    "Variation totale du S/P (points) :",
    round(
        contributions["Variation 2022-2024 (points)"].sum(),
        2
    )
)

In [ ]:
# Identification des variables catégorielles
variables_cat = df.select_dtypes(include=["object", "category"]).columns

for col in variables_cat:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(dropna=False))

In [ ]:
variables_num = [
    "driver_age",
    "vehicle_age",
    "age_driving_licence",
    "vehicle_value",
    "seats",
    "power_to_weight_ratio"
]

display(
    df[variables_num].agg(
        ["min", "median", "max"]
    ).T
)